In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

H5AD_PATH = "/path/atac_peak_matrix_annotated.h5ad"
OUTDIR = "edgeR_pseudobulk_ATAC_PBMC_7major"
os.makedirs(OUTDIR, exist_ok=True)

DONOR_COL = "donor_id"
CELLTYPE_COL = "morocco_Cluster_celltypes_consensus"

PBMC_LABEL = "PBMC_7major"

PBMC_CELLTYPE_LABELS = [
    "Memory T cells",
    "Cytotoxic T/NK cells",
    "B cells",
    "Naive CD8 T cells",
    "Naive CD4 T cells",
    "Monocytes",
    "Dendritic cells"
]

LIFESTYLE_COL = "Lifestyle"
SEX_COL = "Sex"
STIM_COL = "Stimulation"
AGE_COL = "Age"

COUNTS_LAYER = None
MIN_CELLS_PER_DONOR = 10

In [ ]:
# ============================================================
# LOAD
# ============================================================

adata = sc.read_h5ad(H5AD_PATH)
print(adata)

required = [
    DONOR_COL, CELLTYPE_COL, LIFESTYLE_COL,
    SEX_COL, STIM_COL, AGE_COL
]

missing = [x for x in required if x not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing metadata columns: {missing}")

In [ ]:
for x in adata.obs["morocco_Cluster_celltypes_consensus"].unique():
    print(x)

In [ ]:
# ============================================================
# SUBSET TO 7 MAJOR PBMC CELL TYPES
# ============================================================

adata = adata[
    adata.obs[CELLTYPE_COL].astype(str).isin(PBMC_CELLTYPE_LABELS)
].copy()

print("PBMC 7-major-cell-type object:")
print(adata)

print("\nCell types included:")
print(adata.obs[CELLTYPE_COL].value_counts())

In [ ]:
# ============================================================
# GET COUNTS
# ============================================================

if COUNTS_LAYER is None:
    X = adata.X
else:
    X = adata.layers[COUNTS_LAYER]

if sparse.issparse(X):
    X = X.tocsr()
else:
    X = sparse.csr_matrix(X)

In [ ]:
# ============================================================
# SAVE PEAK ANNOTATIONS
# ============================================================

peak_annot = adata.var.copy()
peak_annot = peak_annot.reset_index().rename(columns={"index": "peak_id"})

# Add peak_id explicitly from var_names
peak_annot["peak_id"] = adata.var_names.astype(str)

# Try to standardize best-gene column if present
possible_best_gene_cols = [
    "best_gene",
    "BestGene",
    "bestGene",
    "gene",
    "Gene",
    "nearest_gene",
    "NearestGene",
    "peak_gene",
    "PeakGene"
]

found_gene_cols = [c for c in possible_best_gene_cols if c in peak_annot.columns]

if len(found_gene_cols) > 0:
    peak_annot["best_gene"] = peak_annot[found_gene_cols[0]].astype(str)
else:
    peak_annot["best_gene"] = NA = pd.NA
    print("WARNING: No best-gene column found in adata.var.")
    print("Available adata.var columns:")
    print(list(peak_annot.columns))

peak_annot_file = os.path.join(
    OUTDIR,
    f"{PBMC_LABEL}_ATAC_peak_annotations.tsv"
)

peak_annot.to_csv(
    peak_annot_file,
    sep="\t",
    index=False
)

print("Saved peak annotations:")
print(peak_annot_file)

In [ ]:
# ============================================================
# PREPARE METADATA
# ============================================================

obs = adata.obs.copy()

obs = obs.dropna(
    subset=[
        DONOR_COL, LIFESTYLE_COL,
        SEX_COL, STIM_COL, AGE_COL
    ]
)

adata = adata[obs.index].copy()
X = X[adata.obs_names.isin(obs.index), :]

obs = adata.obs.copy()

for col in [DONOR_COL, LIFESTYLE_COL, SEX_COL, STIM_COL]:
    obs[col] = obs[col].astype(str)

obs["pb_sample_id"] = (
    obs[DONOR_COL].astype(str) + "__" +
    obs[STIM_COL].astype(str) + "__" +
    obs[SEX_COL].astype(str)
)

In [ ]:
# ============================================================
# PSEUDOBULK BY DONOR × STIM × SEX
# ============================================================

pb_ids = obs["pb_sample_id"].values
unique_pb_ids = pd.Index(pd.unique(pb_ids))

pb_counts = []
pb_meta_rows = []

for pb in unique_pb_ids:
    idx = np.where(pb_ids == pb)[0]
    n_cells = len(idx)

    if n_cells < MIN_CELLS_PER_DONOR:
        continue

    summed = np.asarray(X[idx, :].sum(axis=0)).ravel()

    sub = obs.iloc[idx]

    pb_counts.append(summed)

    pb_meta_rows.append({
        "pb_sample_id": pb,
        "donor_id": sub[DONOR_COL].iloc[0],
        "celltype_pool": PBMC_LABEL,
        "Lifestyle": sub[LIFESTYLE_COL].iloc[0],
        "Sex": sub[SEX_COL].iloc[0],
        "Stimulation": sub[STIM_COL].iloc[0],
        "Age": sub[AGE_COL].iloc[0],
        "n_cells": n_cells,
        "library_size": summed.sum()
    })

pb_counts = np.vstack(pb_counts)
pb_meta = pd.DataFrame(pb_meta_rows)

counts_df = pd.DataFrame(
    pb_counts.T,
    index=adata.var_names.astype(str),
    columns=pb_meta["pb_sample_id"].astype(str)
)

In [ ]:
# ============================================================
# SAVE FULL PSEUDOBULK
# ============================================================

counts_df.to_csv(
    os.path.join(OUTDIR, f"{PBMC_LABEL}_ATAC_pseudobulk_counts_all.tsv"),
    sep="\t"
)

pb_meta.to_csv(
    os.path.join(OUTDIR, f"{PBMC_LABEL}_ATAC_pseudobulk_metadata_all.tsv"),
    sep="\t",
    index=False
)


In [ ]:
# ============================================================
# SAVE FOUR COMPARISON MATRICES
# ============================================================

comparisons = [
    ("Unstim", "Male"),
    ("Unstim", "Female"),
    ("LPS", "Male"),
    ("LPS", "Female")
]

for stim, sex in comparisons:

    tag = f"{stim}_{sex}"

    meta_sub = pb_meta[
        (pb_meta["Stimulation"].astype(str) == stim) &
        (pb_meta["Sex"].astype(str) == sex)
    ].copy()

    meta_sub["Lifestyle"] = meta_sub["Lifestyle"].astype(str).str.upper()

    meta_sub = meta_sub[
        meta_sub["Lifestyle"].isin(["URBAN", "RURAL"])
    ].copy()

    sample_ids = meta_sub["pb_sample_id"].astype(str).tolist()

    counts_sub = counts_df[sample_ids].copy()

    counts_sub.to_csv(
        os.path.join(OUTDIR, f"{PBMC_LABEL}_ATAC_counts_{tag}.tsv"),
        sep="\t"
    )

    meta_sub.to_csv(
        os.path.join(OUTDIR, f"{PBMC_LABEL}_ATAC_metadata_{tag}.tsv"),
        sep="\t",
        index=False
    )

    print("\n", tag)
    print("counts:", counts_sub.shape)
    print("metadata:", meta_sub.shape)
    print(meta_sub["Lifestyle"].value_counts())

# ============================================================
# SAVE SUMMARY
# ============================================================

summary = (
    pb_meta
    .assign(Lifestyle=pb_meta["Lifestyle"].astype(str).str.upper())
    .groupby(["Stimulation", "Sex", "Lifestyle"])
    .agg(
        n_donors=("donor_id", "nunique"),
        median_cells=("n_cells", "median"),
        median_library_size=("library_size", "median")
    )
    .reset_index()
)

summary.to_csv(
    os.path.join(OUTDIR, f"{PBMC_LABEL}_ATAC_pseudobulk_summary.tsv"),
    sep="\t",
    index=False
)

print("\nDone.")